# 05 — Final training and export (CNN pré-treinada)

## Objetivo
Executar o treino real de uma CNN pré-treinada para classificação de lesões cutâneas, com foco em:
- reprodutibilidade;
- configuração centralizada;
- avaliação clara em validação e teste;
- geração de artefatos consistentes para as próximas etapas.

## Papel deste notebook na trilha
- O Notebook 04 entrega uma **seleção preliminar/baseline de referência**.
- Este Notebook 05 executa a trilha de **treino real com CNN**, mais forte que o baseline linear.
- A promoção de um modelo a “oficial” deve ser explícita, reproduzível e baseada em artefatos salvos.

## Entradas esperadas
- `data/processed/train.csv`
- `data/processed/val.csv`
- `data/processed/test.csv`
- `data/processed/label_map.json`
- preferencialmente `data/processed/target_config_effective.json`
- opcionalmente artefatos do Notebook 04:
  - `reports/experiments_configs.json`
  - `reports/selection_decision.md`

## Saídas esperadas
- checkpoint do melhor modelo
- histórico de treino
- logits/labels de val/test
- curvas e confusion matrices
- config do treino
- resumo de métricas

## Regras desta versão
- execução de cima para baixo em kernel limpo
- um único bloco oficial de configuração
- preferência pelo contrato `target_config_effective.json`
- baseline do Notebook 04 tratado como **referência**, não como decisão final

In [ ]:
# =========================
# 05_train_real.ipynb - Optional cell
# CUDA/GPU smoke check. This does not block CPU execution.
# =========================

import torch

print("torch:", torch.__version__)
cuda_available = torch.cuda.is_available()
print("cuda available:", cuda_available)

if cuda_available:
    print("gpu:", torch.cuda.get_device_name(0))
    print("cap:", torch.cuda.get_device_capability(0))

    x = torch.randn(1024, 1024, device="cuda")
    y = x @ x
    torch.cuda.synchronize()
    print("OK - CUDA matmul:", y.shape)
else:
    print("CUDA is unavailable; the notebook will continue on CPU when applicable.")


In [ ]:
# =========================
# 05_train_real.ipynb - Cell 01
# Imports + seed + environment + central run knobs
# =========================

from __future__ import annotations

import os
import json
import math
import time
import random
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, Tuple, List, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from PIL import Image
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# ============================================================
# Run knobs
# ============================================================

EXPERIMENT_SEED = 44
TRAINING_PRESET_NAME = "resnet50_a"
RUN_NOTES = "v2-selection-phase"

# Operational candidate chosen after the multi-seed comparison.
OPERATIONAL_CANDIDATE_EXP_NAME = "cls_resnet50_img224_seed44_20260608_123940"

LOW_CONFIDENCE_THRESHOLD = 0.50
AMBIGUITY_GAP_THRESHOLD = 0.10

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(EXPERIMENT_SEED)

REPRO_SEED = int(EXPERIMENT_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    except Exception:
        pass

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("REPRO_SEED:", REPRO_SEED)
print("TRAINING_PRESET_NAME:", TRAINING_PRESET_NAME)
print("OPERATIONAL_CANDIDATE_EXP_NAME:", OPERATIONAL_CANDIDATE_EXP_NAME)
print("RUN_NOTES:", RUN_NOTES)
print("DEVICE:", DEVICE)
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# =========================
# 05_train_real.ipynb — Célula 02
# PROJECT_ROOT + paths + contratos
# =========================

def _looks_like_repo_root(p: Path) -> bool:
    return (
        (p / "data" / "raw" / "lesions" / "images").exists()
        and (p / "data" / "processed").exists()
        and (p / "reports").exists()
    )

def _normalize_repo_candidate(p: Path) -> Optional[Path]:
    p = p.expanduser().resolve()
    if _looks_like_repo_root(p):
        return p
    if _looks_like_repo_root(p / "pimple"):
        return (p / "pimple").resolve()
    return None

def find_project_root_robust() -> Path:
    env_root = os.environ.get("PIMPLE_PROJECT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        normalized = _normalize_repo_candidate(Path(env_root))
        if normalized is not None:
            return normalized
        raise FileNotFoundError(
            f"[ERRO] Env PROJECT_ROOT/PIMPLE_PROJECT_ROOT aponta para {env_root}, "
            "mas não parece ser a raiz do repo pimple nem o diretório pai que contém a pasta pimple."
        )

    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        normalized = _normalize_repo_candidate(base)
        if normalized is not None:
            return normalized

    raise FileNotFoundError(
        "Não consegui localizar a raiz do repositório pimple.\n"
        "Dica: defina os.environ['PIMPLE_PROJECT_ROOT'] = r'CAMINHO_PARA_O_REPO_PIMPLE' e rode de novo."
    )

PROJECT_ROOT = find_project_root_robust()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "lesions"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
PLOTS_DIR = REPORTS_DIR / "plots"

TRAIN_CSV = PROCESSED_DIR / "train.csv"
VAL_CSV   = PROCESSED_DIR / "val.csv"
TEST_CSV  = PROCESSED_DIR / "test.csv"
LABEL_MAP_PATH = PROCESSED_DIR / "label_map.json"

TARGET_CFG_EFFECTIVE_PATH = PROCESSED_DIR / "target_config_effective.json"
TARGET_CFG_PATH = PROCESSED_DIR / "target_config.json"

NB04_EXPERIMENTS_CONFIGS_PATH = REPORTS_DIR / "experiments_configs.json"
NB04_SELECTION_DECISION_PATH = REPORTS_DIR / "selection_decision.md"

for p in [PROCESSED_DIR, REPORTS_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists(), f"Rode o Notebook 02 antes. Não achei: {TRAIN_CSV}"
assert VAL_CSV.exists(),   f"Rode o Notebook 02 antes. Não achei: {VAL_CSV}"
assert TEST_CSV.exists(),  f"Rode o Notebook 02 antes. Não achei: {TEST_CSV}"
assert LABEL_MAP_PATH.exists(), f"Rode o Notebook 02 antes. Não achei: {LABEL_MAP_PATH}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR     :", RAW_DIR)
print("PROCESSED   :", PROCESSED_DIR)
print("REPORTS     :", REPORTS_DIR)
print("target_config_effective existe?", TARGET_CFG_EFFECTIVE_PATH.exists())
print("target_config existe?          ", TARGET_CFG_PATH.exists())
print("NB04 experiments_configs?      ", NB04_EXPERIMENTS_CONFIGS_PATH.exists())
print("NB04 selection_decision?       ", NB04_SELECTION_DECISION_PATH.exists())

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 03
# label_map + target_config efetivo + contexto do NB04
# =========================

label_map = json.loads(LABEL_MAP_PATH.read_text(encoding="utf-8"))
index_to_label = list(label_map["index_to_label"])
label_to_index = dict(label_map["label_to_index"])
NUM_CLASSES = len(index_to_label)

if TARGET_CFG_EFFECTIVE_PATH.exists():
    TARGET_CFG_RESOLVED_PATH = TARGET_CFG_EFFECTIVE_PATH
elif TARGET_CFG_PATH.exists():
    TARGET_CFG_RESOLVED_PATH = TARGET_CFG_PATH
else:
    raise FileNotFoundError(
        "Nenhum target_config encontrado. Esperado: "
        f"{TARGET_CFG_EFFECTIVE_PATH} ou {TARGET_CFG_PATH}"
    )

target_config = json.loads(TARGET_CFG_RESOLVED_PATH.read_text(encoding="utf-8"))
target_definition = target_config.get("target_definition", {})
mask_policy = target_config.get("mask_policy", {})
image_col = target_definition.get("image_col", "image_path")
meta_cols = target_definition.get("meta_cols", [])

nb04_context: Dict[str, Any] = {}
if NB04_EXPERIMENTS_CONFIGS_PATH.exists():
    try:
        nb04_context = json.loads(NB04_EXPERIMENTS_CONFIGS_PATH.read_text(encoding="utf-8"))
    except Exception:
        nb04_context = {}

nb04_selection_mode = nb04_context.get("meta", {}).get("selection_mode", "")
nb04_note = nb04_context.get("meta", {}).get("note", "")

print("NUM_CLASSES:", NUM_CLASSES)
print("CLASSES    :", index_to_label)
print("TARGET_CFG :", TARGET_CFG_RESOLVED_PATH.relative_to(PROJECT_ROOT))
print("image_col  :", image_col)
print("meta_cols  :", meta_cols)
print("mask_policy:", mask_policy)
print("NB04 selection_mode:", nb04_selection_mode or "(não disponível)")
if nb04_note:
    print("NB04 note:", nb04_note)

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 04
# Carregar splits + sanity checks (colunas / paths / ranges)
# =========================

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

required_cols = {"image_path", "target_index"}
for name, df_ in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = required_cols - set(df_.columns)
    assert not missing, f"{name} está sem colunas {missing}. Colunas atuais: {list(df_.columns)}"
    assert df_["target_index"].between(0, NUM_CLASSES - 1).all(), f"{name}: target_index fora do range [0..{NUM_CLASSES-1}]"

def check_paths(df_: pd.DataFrame, n: int = 8, sample_n: int = 250) -> None:
    bad = []
    for _, row in df_.head(sample_n).iterrows():
        p = PROJECT_ROOT / str(row["image_path"])
        if not p.exists():
            bad.append(str(row["image_path"]))
            if len(bad) >= n:
                break
    if bad:
        raise FileNotFoundError(f"Alguns image_path não existem (exemplos): {bad}")
    print(f"OK — paths existem (amostra de {min(sample_n, len(df_))}).")

check_paths(train_df)
check_paths(val_df)
check_paths(test_df)

print("sizes:", len(train_df), len(val_df), len(test_df))
print("\nDistribuição (train):")
print(train_df["target_index"].value_counts().sort_index().rename(index=lambda i: index_to_label[i]))


In [ ]:
# =========================
# 05_train_real.ipynb — Célula 05
# Visual rápido: 9 imagens do treino (sanity)
# =========================

def show_grid(df_: pd.DataFrame, n: int = 9) -> None:
    n = min(n, len(df_))
    idxs = np.random.choice(len(df_), size=n, replace=False)
    cols = int(math.sqrt(n))
    rows = int(math.ceil(n / cols))

    plt.figure(figsize=(10, 10))
    for i, ix in enumerate(idxs, start=1):
        row = df_.iloc[ix]
        img_path = PROJECT_ROOT / str(row["image_path"])
        y = int(row["target_index"])
        img = Image.open(img_path).convert("RGB")

        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(index_to_label[y])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_grid(train_df, n=9)


In [ ]:
# =========================
# 05_train_real.ipynb — Célula 06
# Preset central + transforms + dataset
# =========================

import torchvision.transforms as T

PRESETS: Dict[str, Dict[str, Any]] = {
    "resnet50_a": {
        "model_name": "resnet50",
        "img_size": 224,
        "batch_size": 128,
        "epochs": 18,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.05,
        "loss_type": "ce",
        "focal_gamma": 2.0,
        "use_class_weights": True,
        "use_weighted_sampler": False,
        "safe_dataloader": True,
        "num_workers_fast": 4,
        "use_random_erasing": False,
        "notes": "Preset estável para ResNet50.",
    },
    "mobilenet_v3_large_a": {
        "model_name": "mobilenet_v3_large",
        "img_size": 224,
        "batch_size": 192,
        "epochs": 18,
        "lr": 3e-4,
        "weight_decay": 5e-5,
        "label_smoothing": 0.05,
        "loss_type": "ce",
        "focal_gamma": 1.5,
        "use_class_weights": True,
        "use_weighted_sampler": False,
        "safe_dataloader": True,
        "num_workers_fast": 4,
        "use_random_erasing": False,
        "notes": "Preset estável para MobileNetV3-Large.",
    },
    "efficientnet_b0_a": {
        "model_name": "efficientnet_b0",
        "img_size": 224,
        "batch_size": 128,
        "epochs": 18,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.05,
        "loss_type": "ce",
        "focal_gamma": 1.5,
        "use_class_weights": True,
        "use_weighted_sampler": False,
        "safe_dataloader": True,
        "num_workers_fast": 4,
        "use_random_erasing": False,
        "notes": "Preset estável para EfficientNet-B0.",
    },
}

assert TRAINING_PRESET_NAME in PRESETS, f"Preset inválido: {TRAINING_PRESET_NAME}"

cfg = dict(PRESETS[TRAINING_PRESET_NAME])

TRAINING_PRESET = TRAINING_PRESET_NAME
MODEL_NAME = cfg["model_name"]
IMG_SIZE = int(cfg["img_size"])
BATCH_SIZE = int(cfg["batch_size"])
EPOCHS = int(cfg["epochs"])
LR = float(cfg["lr"])
WEIGHT_DECAY = float(cfg["weight_decay"])
LABEL_SMOOTHING = float(cfg["label_smoothing"])
LOSS_TYPE = str(cfg["loss_type"])
FOCAL_GAMMA = float(cfg["focal_gamma"])
USE_CLASS_WEIGHTS = bool(cfg["use_class_weights"])
USE_WEIGHTED_SAMPLER = bool(cfg["use_weighted_sampler"])
SAFE_DATALOADER = bool(cfg["safe_dataloader"])
NUM_WORKERS_FAST = int(cfg["num_workers_fast"])
USE_RANDOM_ERASING = bool(cfg["use_random_erasing"])
PRESET_NOTES = str(cfg["notes"])

assert not (USE_CLASS_WEIGHTS and USE_WEIGHTED_SAMPLER), (
    "Escolha apenas uma estratégia principal de imbalance: "
    "class_weights OU weighted_sampler."
)

train_tfms_list = [
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),
    T.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10,
        hue=0.02,
    ),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
]

if USE_RANDOM_ERASING:
    train_tfms_list.append(
        T.RandomErasing(
            p=0.25,
            scale=(0.02, 0.10),
            ratio=(0.3, 3.3),
            value="random",
        )
    )

train_tfms = T.Compose(train_tfms_list)

eval_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

class LesionClsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, project_root: Path, tfms: T.Compose):
        self.df = df.reset_index(drop=True)
        self.root = project_root
        self.tfms = tfms

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = self.root / str(row["image_path"])
        y = int(row["target_index"])

        img = Image.open(img_path).convert("RGB")
        x = self.tfms(img)
        return x, y

train_ds = LesionClsDataset(train_df, PROJECT_ROOT, train_tfms)
val_ds   = LesionClsDataset(val_df, PROJECT_ROOT, eval_tfms)
test_ds  = LesionClsDataset(test_df, PROJECT_ROOT, eval_tfms)

print("TRAINING_PRESET :", TRAINING_PRESET)
print("PRESET_NOTES    :", PRESET_NOTES)
print("MODEL_NAME      :", MODEL_NAME)
print("IMG_SIZE        :", IMG_SIZE)
print("BATCH_SIZE      :", BATCH_SIZE)
print("EPOCHS          :", EPOCHS)
print("LOSS_TYPE       :", LOSS_TYPE)
print("USE_CLASS_WEIGHTS   :", USE_CLASS_WEIGHTS)
print("USE_WEIGHTED_SAMPLER:", USE_WEIGHTED_SAMPLER)
print("USE_RANDOM_ERASING  :", USE_RANDOM_ERASING)
print("LOW_CONFIDENCE_THRESHOLD :", LOW_CONFIDENCE_THRESHOLD)
print("AMBIGUITY_GAP_THRESHOLD  :", AMBIGUITY_GAP_THRESHOLD)
print("OK — datasets:", len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 07
# Dataloader + class imbalance
# =========================

from torch.utils.data import DataLoader, WeightedRandomSampler

if SAFE_DATALOADER:
    NUM_WORKERS = 0
    PREFETCH = None
    PERSISTENT = False
else:
    NUM_WORKERS = NUM_WORKERS_FAST
    PREFETCH = 2
    PERSISTENT = False  # mais estável em Windows/Jupyter

train_counts = train_df["target_index"].value_counts().sort_index()
counts = np.array([train_counts.get(i, 0) for i in range(NUM_CLASSES)], dtype=np.float32)

total = counts.sum()
class_weights = total / (NUM_CLASSES * np.clip(counts, 1.0, None))
class_weights = class_weights / class_weights.mean()

print("counts:", {index_to_label[i]: int(counts[i]) for i in range(NUM_CLASSES)})
print("class_weights:", {index_to_label[i]: float(class_weights[i]) for i in range(NUM_CLASSES)})

sampler = None
if USE_WEIGHTED_SAMPLER:
    sample_w = class_weights[train_df["target_index"].values.astype(int)]
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_w, dtype=torch.double),
        num_samples=len(sample_w),
        replacement=True,
    )

def seed_worker(worker_id: int) -> None:
    worker_seed = REPRO_SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(REPRO_SEED)

common = dict(
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=PERSISTENT,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    generator=g,
)

if NUM_WORKERS > 0 and PREFETCH is not None:
    common["prefetch_factor"] = PREFETCH

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=(sampler is None),
    sampler=sampler,
    drop_last=True,
    **common,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **common,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **common,
)

print("OK — dataloaders.")
print("SAFE_DATALOADER:", SAFE_DATALOADER)
print("BATCH_SIZE:", BATCH_SIZE, "| NUM_WORKERS:", NUM_WORKERS)
print("batches train:", len(train_loader), "| val:", len(val_loader), "| test:", len(test_loader))

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 08
# Modelo torchvision + loss + optimizer + scheduler
# =========================

import torchvision
import torchvision.models as tvm

def build_torchvision_model(name: str, num_classes: int) -> nn.Module:
    name = name.lower()

    if name == "resnet50":
        weights = tvm.ResNet50_Weights.DEFAULT
        m = tvm.resnet50(weights=weights)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m

    if name == "efficientnet_b0":
        weights = tvm.EfficientNet_B0_Weights.DEFAULT
        m = tvm.efficientnet_b0(weights=weights)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_f, num_classes)
        return m

    if name == "mobilenet_v3_large":
        weights = tvm.MobileNet_V3_Large_Weights.DEFAULT
        m = tvm.mobilenet_v3_large(weights=weights)
        in_f = m.classifier[3].in_features
        m.classifier[3] = nn.Linear(in_f, num_classes)
        return m

    raise ValueError(
        f"MODEL_NAME inválido: {name}. "
        "Use resnet50 / efficientnet_b0 / mobilenet_v3_large."
    )

model = build_torchvision_model(MODEL_NAME, NUM_CLASSES).to(DEVICE)

if DEVICE.type == "cuda":
    model = model.to(memory_format=torch.channels_last)

w = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE) if USE_CLASS_WEIGHTS else None

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        logp = F.log_softmax(logits, dim=1)
        p = torch.exp(logp)
        pt = p.gather(1, target.view(-1, 1)).squeeze(1)
        logpt = logp.gather(1, target.view(-1, 1)).squeeze(1)
        loss = -((1.0 - pt) ** self.gamma) * logpt
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean()

if LOSS_TYPE == "ce":
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
elif LOSS_TYPE == "focal":
    criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=w)
else:
    raise ValueError("LOSS_TYPE inválido. Use 'ce' ou 'focal'.")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)

# API nova do PyTorch para AMP
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

print("OK — torchvision model carregado:", MODEL_NAME)
print("torchvision:", torchvision.__version__)
print("LR:", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("EPOCHS:", EPOCHS)
print("LABEL_SMOOTHING:", LABEL_SMOOTHING)
print("LOSS_TYPE:", LOSS_TYPE)

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 09
# Helpers de treino/avaliação + GPU stats
# =========================

from tqdm.auto import tqdm

def _gpu_mem_gb() -> float:
    if DEVICE.type != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / (1024**3)

@torch.no_grad()
def predict_logits(model: nn.Module, loader: DataLoader, desc: str = "val") -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_logits, all_y = [], []
    pbar = tqdm(loader, desc=desc, leave=False)

    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if DEVICE.type == "cuda":
            x = x.to(memory_format=torch.channels_last)

        logits = model(x)
        all_logits.append(logits.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())

    return np.concatenate(all_logits, axis=0), np.concatenate(all_y, axis=0)

def eval_metrics_from_logits(logits: np.ndarray, y_true: np.ndarray) -> Dict[str, float]:
    y_pred = logits.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    }

def train_one_epoch(model: nn.Module, loader: DataLoader, epoch: int, epochs: int) -> float:
    model.train()
    running = 0.0
    n = 0

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    # debug do primeiro batch
    t_load0 = time.time()
    it = iter(loader)
    x0, y0 = next(it)
    t_load1 = time.time()
    print(f"Epoch {epoch}: primeiro batch carregado em {t_load1 - t_load0:.2f}s")

    def _step_batch(x, y):
        nonlocal running, n

        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if DEVICE.type == "cuda":
            x = x.to(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += float(loss.item()) * x.size(0)
        n += x.size(0)

        return float(loss.item())

    t0 = time.time()
    _ = _step_batch(x0, y0)

    pbar = tqdm(it, total=len(loader) - 1, desc=f"train {epoch}/{epochs}", leave=False)
    for x, y in pbar:
        _ = _step_batch(x, y)
        avg_loss = running / max(n, 1)
        lr_now = optimizer.param_groups[0]["lr"]

        if DEVICE.type == "cuda":
            mem_gb = _gpu_mem_gb()
            pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{lr_now:.2e}", mem=f"{mem_gb:.2f}GB")
        else:
            pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{lr_now:.2e}")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    dt = time.time() - t0
    sps = n / max(dt, 1e-9)
    train_loss = running / max(n, 1)

    print(
        f"Epoch {epoch}: train_loss={train_loss:.4f} | "
        f"{sps:.1f} samples/s | {dt/60:.2f} min | peak_mem={_gpu_mem_gb():.2f}GB"
    )

    return train_loss

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 10
# Train loop + early stopping + checkpoint rico
# =========================

def get_git_commit(project_root: Path) -> str:
    try:
        out = subprocess.check_output(
            ["git", "-C", str(project_root), "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL,
        )
        return out.decode("utf-8").strip()
    except Exception:
        return ""

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXP_NAME = f"cls_{MODEL_NAME}_img{IMG_SIZE}_seed{REPRO_SEED}_{RUN_ID}"

MODEL_DIR = PROJECT_ROOT / "models" / "05_train_real" / EXP_NAME
RUN_PLOTS_DIR = PLOTS_DIR / "05_train" / EXP_NAME
RUN_ART_DIR = PROCESSED_DIR / "05_runs" / EXP_NAME

for p in [MODEL_DIR, RUN_PLOTS_DIR, RUN_ART_DIR]:
    p.mkdir(parents=True, exist_ok=True)

best_f1 = -1.0
best_epoch = -1
patience = 3
no_improve = 0

history: List[Dict[str, float]] = []
epoch_times: List[float] = []

print("=== START TRAIN LOOP ===")
print("EXP_NAME:", EXP_NAME)
print("git_commit:", get_git_commit(PROJECT_ROOT))
print("DEVICE:", DEVICE)
print("TRAINING_PRESET:", TRAINING_PRESET)
print("RUN_NOTES:", RUN_NOTES)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
    t_epoch = time.time()

    tr_loss = train_one_epoch(model, train_loader, epoch, EPOCHS)
    scheduler.step()

    val_logits, val_y = predict_logits(model, val_loader, desc="val")
    val_metrics = eval_metrics_from_logits(val_logits, val_y)

    row = {
        "epoch": int(epoch),
        "train_loss": float(tr_loss),
        "val_accuracy": float(val_metrics["accuracy"]),
        "val_f1_macro": float(val_metrics["f1_macro"]),
        "lr": float(optimizer.param_groups[0]["lr"]),
    }
    history.append(row)

    dt_epoch = time.time() - t_epoch
    epoch_times.append(dt_epoch)
    avg_epoch = sum(epoch_times) / len(epoch_times)
    eta = (EPOCHS - epoch) * avg_epoch

    print(
        f"[{epoch:02d}/{EPOCHS}] "
        f"loss={tr_loss:.4f} | "
        f"val_acc={row['val_accuracy']:.4f} | "
        f"val_f1m={row['val_f1_macro']:.4f} | "
        f"lr={row['lr']:.2e} | "
        f"epoch={dt_epoch/60:.2f} min | ETA={eta/60:.1f} min"
    )

    if row["val_f1_macro"] > best_f1 + 1e-6:
        best_f1 = row["val_f1_macro"]
        best_epoch = epoch
        no_improve = 0

        ckpt = {
            "exp_name": EXP_NAME,
            "run_id": RUN_ID,
            "model_name": MODEL_NAME,
            "img_size": int(IMG_SIZE),
            "num_classes": int(NUM_CLASSES),
            "index_to_label": index_to_label,
            "state_dict": model.state_dict(),
            "epoch": int(epoch),
            "val_accuracy": float(row["val_accuracy"]),
            "val_f1_macro": float(best_f1),
            "seed": int(REPRO_SEED),
            "loss_type": LOSS_TYPE,
            "label_smoothing": float(LABEL_SMOOTHING),
            "use_class_weights": bool(USE_CLASS_WEIGHTS),
            "use_weighted_sampler": bool(USE_WEIGHTED_SAMPLER),
            "lr": float(LR),
            "weight_decay": float(WEIGHT_DECAY),
            "preset_name": TRAINING_PRESET,
            "preset_notes": PRESET_NOTES,
            "run_notes": RUN_NOTES,
            "target_config_source": str(TARGET_CFG_RESOLVED_PATH.relative_to(PROJECT_ROOT)),
            "git_commit": get_git_commit(PROJECT_ROOT),
            "selection_policy": "best_checkpoint_by_val_f1_macro",
        }
        torch.save(ckpt, MODEL_DIR / "best.pt")
        print(f"Checkpoint salvo: best.pt (epoch={epoch}, f1m={best_f1:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(
                f"Early stopping: sem melhora em {patience} épocas. "
                f"Best epoch={best_epoch} | best_f1={best_f1:.4f}"
            )
            break

elapsed = time.time() - t0
print("\nTreino finalizado.")
print("elapsed(min):", round(elapsed / 60, 2))
print("BEST:", best_epoch, best_f1)

hist_df = pd.DataFrame(history)
hist_df.to_csv(RUN_ART_DIR / "train_history.csv", index=False, encoding="utf-8")

print("OK — train_history.csv salvo.")

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 11
# Plots: loss e métricas por época (salvar)
# =========================

def plot_curve(x, y, title, xlabel, ylabel, save_path: Path) -> None:
    plt.figure(figsize=(7, 4))
    plt.plot(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=160)
    plt.close()

epochs = hist_df["epoch"].values
plot_curve(epochs, hist_df["train_loss"].values, "Train Loss", "epoch", "loss", RUN_PLOTS_DIR / "curve_train_loss.png")
plot_curve(epochs, hist_df["val_accuracy"].values, "Val Accuracy", "epoch", "accuracy", RUN_PLOTS_DIR / "curve_val_accuracy.png")
plot_curve(epochs, hist_df["val_f1_macro"].values, "Val Macro-F1", "epoch", "f1_macro", RUN_PLOTS_DIR / "curve_val_f1_macro.png")

print("OK — curvas salvas em:", RUN_PLOTS_DIR)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List

def plot_confusion(cm: np.ndarray, class_names: List[str], title: str, save_path: Path, normalize: bool = False) -> None:
    cm_plot = cm.astype(np.float32)
    if normalize:
        cm_plot = cm_plot / np.clip(cm_plot.sum(axis=1, keepdims=True), 1e-9, None)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot)
    plt.title(title)
    plt.xlabel("Pred")
    plt.ylabel("True")
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm_plot[i, j]
            s = f"{val:.2f}" if normalize else f"{int(cm[i,j])}"
            plt.text(j, i, s, ha="center", va="center", fontsize=8)

    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=160)
    plt.close()


In [ ]:
# =========================
# 05_train_real.ipynb — Célula 12
# Avaliar BEST em val/test + salvar logits/labels + error analysis enriquecido
# =========================

best_path = MODEL_DIR / "best.pt"
assert best_path.exists(), f"Checkpoint não encontrado: {best_path}"

ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# ------------------------------------------------------------
# VAL
# ------------------------------------------------------------
val_logits, val_y = predict_logits(model, val_loader, desc="val")
val_pred = val_logits.argmax(axis=1)
val_metrics = eval_metrics_from_logits(val_logits, val_y)
val_cm = confusion_matrix(val_y, val_pred, labels=list(range(NUM_CLASSES)))

print("VAL metrics:", val_metrics)

plot_confusion(
    val_cm,
    class_names=index_to_label,
    title=f"VAL Confusion (best epoch={ckpt['epoch']} f1m={ckpt['val_f1_macro']:.3f})",
    save_path=RUN_PLOTS_DIR / "cm_val.png",
    normalize=False,
)
plot_confusion(
    val_cm,
    class_names=index_to_label,
    title="VAL Confusion (normalized)",
    save_path=RUN_PLOTS_DIR / "cm_val_norm.png",
    normalize=True,
)

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------
test_logits, test_y = predict_logits(model, test_loader, desc="test")
test_pred = test_logits.argmax(axis=1)
test_metrics = eval_metrics_from_logits(test_logits, test_y)
test_cm = confusion_matrix(test_y, test_pred, labels=list(range(NUM_CLASSES)))

print("TEST metrics:", test_metrics)

plot_confusion(
    test_cm,
    class_names=index_to_label,
    title="TEST Confusion",
    save_path=RUN_PLOTS_DIR / "cm_test.png",
    normalize=False,
)
plot_confusion(
    test_cm,
    class_names=index_to_label,
    title="TEST Confusion (normalized)",
    save_path=RUN_PLOTS_DIR / "cm_test_norm.png",
    normalize=True,
)

# ------------------------------------------------------------
# Logits/labels
# ------------------------------------------------------------
np.save(RUN_ART_DIR / "val_logits.npy", val_logits)
np.save(RUN_ART_DIR / "val_labels.npy", val_y)
np.save(RUN_ART_DIR / "test_logits.npy", test_logits)
np.save(RUN_ART_DIR / "test_labels.npy", test_y)

# ------------------------------------------------------------
# Probabilidades e sinais de incerteza
# ------------------------------------------------------------
val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()

def top2_from_probs(probs: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    order = np.argsort(-probs, axis=1)
    top1_idx = order[:, 0]
    top2_idx = order[:, 1]
    return top1_idx, top2_idx

val_top1_idx, val_top2_idx = top2_from_probs(val_probs)
test_top1_idx, test_top2_idx = top2_from_probs(test_probs)

val_top1_score = val_probs[np.arange(len(val_probs)), val_top1_idx]
val_top2_score = val_probs[np.arange(len(val_probs)), val_top2_idx]
test_top1_score = test_probs[np.arange(len(test_probs)), test_top1_idx]
test_top2_score = test_probs[np.arange(len(test_probs)), test_top2_idx]

val_gap = val_top1_score - val_top2_score
test_gap = test_top1_score - test_top2_score

# ------------------------------------------------------------
# Error analysis enriquecido
# ------------------------------------------------------------
error_df = pd.DataFrame({
    "y_true_idx": test_y,
    "y_pred_idx": test_pred,
    "y_true_label": [index_to_label[int(i)] for i in test_y],
    "y_pred_label": [index_to_label[int(i)] for i in test_pred],
    "top1_label": [index_to_label[int(i)] for i in test_top1_idx],
    "top2_label": [index_to_label[int(i)] for i in test_top2_idx],
    "top1_score": test_top1_score,
    "top2_score": test_top2_score,
    "top1_top2_gap": test_gap,
    "confidence": test_top1_score,
    "is_error": (test_y != test_pred).astype(int),
    "is_low_confidence": (test_top1_score < LOW_CONFIDENCE_THRESHOLD).astype(int),
    "is_ambiguous": (test_gap < AMBIGUITY_GAP_THRESHOLD).astype(int),
})

test_df_export = test_df.reset_index(drop=True).copy()
if len(test_df_export) == len(error_df):
    for col in ["image_path", "target_index", "target_label"]:
        if col in test_df_export.columns:
            error_df[col] = test_df_export[col].values

error_df.to_csv(RUN_ART_DIR / "error_analysis.csv", index=False, encoding="utf-8")

test_metrics_payload = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "exp_name": EXP_NAME,
    "model_name": MODEL_NAME,
    "preset_name": TRAINING_PRESET,
    "seed": int(REPRO_SEED),
    "val": val_metrics,
    "test": test_metrics,
    "best_epoch": int(ckpt["epoch"]),
    "classes": index_to_label,
    "uncertainty_thresholds": {
        "low_confidence_threshold": float(LOW_CONFIDENCE_THRESHOLD),
        "ambiguity_gap_threshold": float(AMBIGUITY_GAP_THRESHOLD),
    },
    "uncertainty_counts_test": {
        "n_low_confidence": int((error_df["is_low_confidence"] == 1).sum()),
        "n_ambiguous": int((error_df["is_ambiguous"] == 1).sum()),
        "n_low_confidence_and_error": int(((error_df["is_low_confidence"] == 1) & (error_df["is_error"] == 1)).sum()),
        "n_ambiguous_and_error": int(((error_df["is_ambiguous"] == 1) & (error_df["is_error"] == 1)).sum()),
    },
}
(RUN_ART_DIR / "test_metrics.json").write_text(
    json.dumps(test_metrics_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("OK — logits/labels salvos em:", RUN_ART_DIR)
print("OK — error_analysis.csv enriquecido salvo.")
print("OK — test_metrics.json salvo.")

In [ ]:
# =========================
# 05_train_real.ipynb — Célula 13
# Classification report + export formal + summary final do run
# =========================

report_val = classification_report(val_y, val_pred, target_names=index_to_label, digits=4)
report_test = classification_report(test_y, test_pred, target_names=index_to_label, digits=4)

(RUN_ART_DIR / "classification_report_val.txt").write_text(report_val, encoding="utf-8")
(RUN_ART_DIR / "classification_report_test.txt").write_text(report_test, encoding="utf-8")

train_config = {
    "exp_name": EXP_NAME,
    "run_id": RUN_ID,
    "seed": int(REPRO_SEED),
    "git_commit": get_git_commit(PROJECT_ROOT),
    "model_name": MODEL_NAME,
    "preset_name": TRAINING_PRESET,
    "preset_notes": PRESET_NOTES,
    "run_notes": RUN_NOTES,
    "img_size": int(IMG_SIZE),
    "num_classes": int(NUM_CLASSES),
    "classes": index_to_label,
    "batch_size": int(BATCH_SIZE),
    "epochs_requested": int(EPOCHS),
    "epochs_ran": int(hist_df["epoch"].max()),
    "best_epoch": int(ckpt["epoch"]),
    "best_val_f1_macro": float(ckpt["val_f1_macro"]),
    "best_val_accuracy": float(ckpt.get("val_accuracy", val_metrics["accuracy"])),
    "loss_type": LOSS_TYPE,
    "focal_gamma": float(FOCAL_GAMMA),
    "label_smoothing": float(LABEL_SMOOTHING),
    "use_class_weights": bool(USE_CLASS_WEIGHTS),
    "use_weighted_sampler": bool(USE_WEIGHTED_SAMPLER),
    "lr": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "scheduler": "CosineAnnealingLR",
    "device": str(DEVICE),
    "selection_policy": "compare_runs_by_mean_val_f1_macro_across_seeds",
    "inputs": {
        "train_csv": str(TRAIN_CSV.relative_to(PROJECT_ROOT)),
        "val_csv": str(VAL_CSV.relative_to(PROJECT_ROOT)),
        "test_csv": str(TEST_CSV.relative_to(PROJECT_ROOT)),
        "label_map": str(LABEL_MAP_PATH.relative_to(PROJECT_ROOT)),
        "target_config": str(TARGET_CFG_RESOLVED_PATH.relative_to(PROJECT_ROOT)),
        "nb04_experiments_configs": str(NB04_EXPERIMENTS_CONFIGS_PATH.relative_to(PROJECT_ROOT)) if NB04_EXPERIMENTS_CONFIGS_PATH.exists() else "",
        "nb04_selection_decision": str(NB04_SELECTION_DECISION_PATH.relative_to(PROJECT_ROOT)) if NB04_SELECTION_DECISION_PATH.exists() else "",
    },
    "outputs": {
        "model_dir": str(MODEL_DIR.relative_to(PROJECT_ROOT)),
        "run_art_dir": str(RUN_ART_DIR.relative_to(PROJECT_ROOT)),
        "plots_dir": str(RUN_PLOTS_DIR.relative_to(PROJECT_ROOT)),
        "best_checkpoint": str((MODEL_DIR / "best.pt").relative_to(PROJECT_ROOT)),
    },
}

preprocess_config = {
    "type": "torchvision_imagenet_classification",
    "image_size": [int(IMG_SIZE), int(IMG_SIZE)],
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
    "train_augmentations": {
        "horizontal_flip": True,
        "vertical_flip": True,
        "rotation_degrees": 180,
        "color_jitter": {
            "brightness": 0.10,
            "contrast": 0.10,
            "saturation": 0.10,
            "hue": 0.02,
        },
        "random_erasing": bool(USE_RANDOM_ERASING),
    },
}

inference_config = {
    "task": "classification",
    "model_version": EXP_NAME,
    "model_name": MODEL_NAME,
    "n_classes": int(NUM_CLASSES),
    "class_names": index_to_label,
    "input_size": [int(IMG_SIZE), int(IMG_SIZE)],
    "top_k_default": 3,
    "target_config_source": str(TARGET_CFG_RESOLVED_PATH.relative_to(PROJECT_ROOT)),
    "uncertainty": {
        "low_confidence_threshold": float(LOW_CONFIDENCE_THRESHOLD),
        "ambiguity_gap_threshold": float(AMBIGUITY_GAP_THRESHOLD),
    },
}

metrics_summary = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "exp_name": EXP_NAME,
    "preset_name": TRAINING_PRESET,
    "seed": int(REPRO_SEED),
    "val": val_metrics,
    "test": test_metrics,
}

(RUN_ART_DIR / "train_config.json").write_text(
    json.dumps(train_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(RUN_ART_DIR / "preprocess_config.json").write_text(
    json.dumps(preprocess_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(RUN_ART_DIR / "inference_config.json").write_text(
    json.dumps(inference_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(RUN_ART_DIR / "metrics_summary.json").write_text(
    json.dumps(metrics_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

comparison_row = {
    "exp_name": EXP_NAME,
    "run_id": RUN_ID,
    "preset_name": TRAINING_PRESET,
    "model_name": MODEL_NAME,
    "seed": int(REPRO_SEED),
    "img_size": int(IMG_SIZE),
    "batch_size": int(BATCH_SIZE),
    "epochs_ran": int(hist_df["epoch"].max()),
    "best_epoch": int(ckpt["epoch"]),
    "best_val_accuracy": float(val_metrics["accuracy"]),
    "best_val_f1_macro": float(val_metrics["f1_macro"]),
    "test_accuracy": float(test_metrics["accuracy"]),
    "test_f1_macro": float(test_metrics["f1_macro"]),
    "low_confidence_threshold": float(LOW_CONFIDENCE_THRESHOLD),
    "ambiguity_gap_threshold": float(AMBIGUITY_GAP_THRESHOLD),
    "n_test_errors": int((error_df["is_error"] == 1).sum()),
    "n_test_low_confidence": int((error_df["is_low_confidence"] == 1).sum()),
    "n_test_ambiguous": int((error_df["is_ambiguous"] == 1).sum()),
    "run_notes": RUN_NOTES,
    "run_art_dir": str(RUN_ART_DIR.relative_to(PROJECT_ROOT)),
    "best_checkpoint": str((MODEL_DIR / "best.pt").relative_to(PROJECT_ROOT)),
}

(RUN_ART_DIR / "comparison_row.json").write_text(
    json.dumps(comparison_row, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

final_summary_md = f"""# Final Summary — Notebook 05

## Run
- exp_name: `{EXP_NAME}`
- model_name: `{MODEL_NAME}`
- preset_name: `{TRAINING_PRESET}`
- seed: `{REPRO_SEED}`
- run_notes: `{RUN_NOTES}`
- device: `{DEVICE}`

## Contract
- target_config: `{str(TARGET_CFG_RESOLVED_PATH.relative_to(PROJECT_ROOT))}`
- classes: `{", ".join(index_to_label)}`

## Selection policy
- A comparação entre famílias/presets deve ser feita por média de `val_f1_macro` entre seeds.
- O conjunto de teste deve ser usado apenas para fechamento do candidato vencedor.

## Validation
- accuracy: `{val_metrics["accuracy"]:.4f}`
- f1_macro: `{val_metrics["f1_macro"]:.4f}`

## Test
- accuracy: `{test_metrics["accuracy"]:.4f}`
- f1_macro: `{test_metrics["f1_macro"]:.4f}`

## Uncertainty
- low_confidence_threshold: `{LOW_CONFIDENCE_THRESHOLD:.2f}`
- ambiguity_gap_threshold: `{AMBIGUITY_GAP_THRESHOLD:.2f}`
- n_low_confidence_test: `{int((error_df["is_low_confidence"] == 1).sum())}`
- n_ambiguous_test: `{int((error_df["is_ambiguous"] == 1).sum())}`

## Notes
- O baseline do Notebook 04 deve ser tratado como referência.
- Este run representa a trilha de treino real com CNN pré-treinada.
- Este notebook agora exporta artefatos para comparação final entre presets e seeds.
"""
(RUN_ART_DIR / "final_summary.md").write_text(final_summary_md, encoding="utf-8")

print("=== OK — Artefatos finais ===")
print("EXP_NAME   :", EXP_NAME)
print("MODEL_DIR  :", MODEL_DIR)
print("RUN_ART_DIR:", RUN_ART_DIR)
print("PLOTS_DIR  :", RUN_PLOTS_DIR)
print()
print("Arquivos gerados:")
print("- train_config.json")
print("- preprocess_config.json")
print("- inference_config.json")
print("- metrics_summary.json")
print("- test_metrics.json")
print("- error_analysis.csv")
print("- comparison_row.json")
print("- classification_report_val.txt")
print("- classification_report_test.txt")
print("- final_summary.md")
print()
print(">>> Este run já está pronto para entrar na comparação final.")
print(">>> Use este EXP_NAME no Notebook 06 apenas se ele vencer a comparação.")

In [ ]:
# =========================
# 05_train_real.ipynb - Cell 14
# Update consolidated comparison table between runs
# =========================

COMPARISON_CSV = REPORTS_DIR / "final_model_comparison.csv"

row_df = pd.DataFrame([comparison_row])

if COMPARISON_CSV.exists():
    old_df = pd.read_csv(COMPARISON_CSV)
    comparison_df = pd.concat([old_df, row_df], ignore_index=True)
else:
    comparison_df = row_df.copy()

def run_id_key(value: object) -> int:
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    return int(digits) if digits else 0

# Keep only the most recent run for each preset/model/seed.
comparison_df["_run_id_key"] = comparison_df["run_id"].map(run_id_key)
comparison_df = (
    comparison_df
    .sort_values(
        by=["preset_name", "model_name", "seed", "_run_id_key"],
        ascending=[True, True, True, True],
    )
    .drop_duplicates(subset=["preset_name", "model_name", "seed"], keep="last")
    .drop(columns=["_run_id_key"])
    .sort_values(
        by=["preset_name", "model_name", "seed"],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)

comparison_df.to_csv(COMPARISON_CSV, index=False, encoding="utf-8")

print("OK - final_model_comparison.csv updated without preset/model/seed duplicates at:")
print(COMPARISON_CSV)
display(comparison_df)


In [ ]:
# =========================
# 05_train_real.ipynb - Cell 15
# Final selection by mean validation score across seeds
# =========================

COMPARISON_CSV = REPORTS_DIR / "final_model_comparison.csv"
assert COMPARISON_CSV.exists(), f"File not found: {COMPARISON_CSV}"

comparison_df = pd.read_csv(COMPARISON_CSV)

def run_id_key(value: object) -> int:
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    return int(digits) if digits else 0

# Defensive cleanup for older CSVs: selection always operates on unique seeds.
comparison_df["_run_id_key"] = comparison_df["run_id"].map(run_id_key)
comparison_df = (
    comparison_df
    .sort_values(
        by=["preset_name", "model_name", "seed", "_run_id_key"],
        ascending=[True, True, True, True],
    )
    .drop_duplicates(subset=["preset_name", "model_name", "seed"], keep="last")
    .reset_index(drop=True)
)

summary_df = (
    comparison_df
    .groupby(["preset_name", "model_name"], as_index=False)
    .agg(
        n_runs=("seed", "nunique"),
        val_f1_macro_mean=("best_val_f1_macro", "mean"),
        val_f1_macro_std=("best_val_f1_macro", "std"),
        val_accuracy_mean=("best_val_accuracy", "mean"),
        test_f1_macro_mean=("test_f1_macro", "mean"),
        test_accuracy_mean=("test_accuracy", "mean"),
    )
    .sort_values(by=["val_f1_macro_mean", "val_accuracy_mean"], ascending=False)
    .reset_index(drop=True)
)

winner = summary_df.iloc[0].to_dict()

winner_runs = comparison_df[
    (comparison_df["preset_name"] == winner["preset_name"])
    & (comparison_df["model_name"] == winner["model_name"])
].copy()

operational_candidate = (
    winner_runs
    .sort_values(
        by=["best_val_f1_macro", "best_val_accuracy", "_run_id_key"],
        ascending=[False, False, False],
    )
    .iloc[0]
    .to_dict()
)

comparison_df = comparison_df.drop(columns=["_run_id_key"])
comparison_df = comparison_df.sort_values(
    by=["preset_name", "model_name", "seed"],
    ascending=[True, True, True],
).reset_index(drop=True)
comparison_df.to_csv(COMPARISON_CSV, index=False, encoding="utf-8")

SELECTION_MD = REPORTS_DIR / "final_model_selection.md"

summary_table_text = summary_df.round(6).to_string(index=False)

selection_md = (
    "# Final Model Selection - Notebook 05\n\n"
    "## Policy\n"
    "- The champion family is selected by **mean `val_f1_macro`** across unique seeds.\n"
    "- If more than one run exists for the same `preset_name`/`model_name`/`seed`, the most recent run is kept.\n"
    "- The test set is reported only as a final readout for the winner, not as the primary selection criterion.\n\n"
    "## Aggregated results\n\n"
    + summary_table_text
    + "\n\n"
    "## Winner\n"
    f"- preset_name: `{winner['preset_name']}`\n"
    f"- model_name: `{winner['model_name']}`\n"
    f"- n_runs: `{int(winner['n_runs'])}`\n"
    f"- val_f1_macro_mean: `{winner['val_f1_macro_mean']:.6f}`\n"
    f"- val_f1_macro_std: `{0.0 if pd.isna(winner['val_f1_macro_std']) else winner['val_f1_macro_std']:.6f}`\n"
    f"- val_accuracy_mean: `{winner['val_accuracy_mean']:.6f}`\n"
    f"- test_f1_macro_mean: `{winner['test_f1_macro_mean']:.6f}`\n"
    f"- test_accuracy_mean: `{winner['test_accuracy_mean']:.6f}`\n\n"
    "## Operational candidate\n"
    f"- exp_name: `{operational_candidate['exp_name']}`\n"
    f"- seed: `{int(operational_candidate['seed'])}`\n"
    f"- val_f1_macro: `{float(operational_candidate['best_val_f1_macro']):.6f}`\n"
    f"- test_f1_macro: `{float(operational_candidate['test_f1_macro']):.6f}`\n"
    "- selection_basis: best `val_f1_macro` within the winning family after seed deduplication.\n\n"
    "## Notes\n"
    "- Family selection uses validation metrics aggregated across seeds.\n"
    "- Operational candidate selection uses validation metrics inside the winning family.\n"
    "- Test metrics are reported only as final readout and are not part of the primary selection criterion.\n"
    "- Notebook 06 must validate the operational candidate before any active-model promotion.\n"
)

SELECTION_MD.write_text(selection_md, encoding="utf-8")

print("OK - final_model_selection.md saved at:")
print(SELECTION_MD)
display(summary_df)
print("Operational candidate:", operational_candidate["exp_name"])


In [ ]:
# =========================
# 05_train_real.ipynb - Cell 16
# Uncertainty policy and auxiliary metrics
# =========================

UNCERTAINTY_METRICS_CSV = REPORTS_DIR / "uncertainty_metrics.csv"
UNCERTAINTY_POLICY_MD = REPORTS_DIR / "uncertainty_policy.md"

uncertainty_summary = pd.DataFrame([
    {
        "exp_name": EXP_NAME,
        "preset_name": TRAINING_PRESET,
        "seed": int(REPRO_SEED),
        "low_confidence_threshold": float(LOW_CONFIDENCE_THRESHOLD),
        "ambiguity_gap_threshold": float(AMBIGUITY_GAP_THRESHOLD),
        "n_samples_test": int(len(error_df)),
        "n_errors_test": int((error_df["is_error"] == 1).sum()),
        "n_low_confidence_test": int((error_df["is_low_confidence"] == 1).sum()),
        "n_ambiguous_test": int((error_df["is_ambiguous"] == 1).sum()),
        "n_low_confidence_and_error": int(
            ((error_df["is_low_confidence"] == 1) & (error_df["is_error"] == 1)).sum()
        ),
        "n_ambiguous_and_error": int(
            ((error_df["is_ambiguous"] == 1) & (error_df["is_error"] == 1)).sum()
        ),
        "low_confidence_rate": float((error_df["is_low_confidence"] == 1).mean()),
        "ambiguous_rate": float((error_df["is_ambiguous"] == 1).mean()),
    }
])

if UNCERTAINTY_METRICS_CSV.exists():
    old_unc_df = pd.read_csv(UNCERTAINTY_METRICS_CSV)
    uncertainty_df = pd.concat([old_unc_df, uncertainty_summary], ignore_index=True)
    uncertainty_df = uncertainty_df.drop_duplicates(subset=["exp_name"], keep="last")
else:
    uncertainty_df = uncertainty_summary.copy()

uncertainty_df.to_csv(UNCERTAINTY_METRICS_CSV, index=False, encoding="utf-8")

policy_exp_name = globals().get("OPERATIONAL_CANDIDATE_EXP_NAME", EXP_NAME)
if policy_exp_name in set(uncertainty_df["exp_name"].astype(str)):
    policy_row = uncertainty_df[uncertainty_df["exp_name"].astype(str) == policy_exp_name].iloc[-1]
else:
    policy_row = uncertainty_summary.iloc[0]

policy_table_text = pd.DataFrame([policy_row]).round(6).to_string(index=False)

policy_md = (
    "# Uncertainty Policy - Notebook 05\n\n"
    "## Current thresholds\n"
    f"- low_confidence_threshold: `{float(policy_row['low_confidence_threshold']):.2f}`\n"
    f"- ambiguity_gap_threshold: `{float(policy_row['ambiguity_gap_threshold']):.2f}`\n\n"
    "## Current operational candidate\n"
    f"- exp_name: `{policy_row['exp_name']}`\n"
    f"- preset_name: `{policy_row['preset_name']}`\n"
    f"- seed: `{int(policy_row['seed'])}`\n\n"
    "## Interpretation\n"
    f"- **Low confidence**: `top1_score < {float(policy_row['low_confidence_threshold']):.2f}`\n"
    f"- **Ambiguous prediction**: `(top1_score - top2_score) < {float(policy_row['ambiguity_gap_threshold']):.2f}`\n\n"
    "## Candidate metrics\n\n"
    + policy_table_text
    + "\n\n"
    "## Notes\n"
    "- This policy is still heuristic and should be treated as an initial baseline.\n"
    "- It supports future product rules such as caution notices and possible out-of-domain warnings.\n"
)

UNCERTAINTY_POLICY_MD.write_text(policy_md, encoding="utf-8")

print("OK - uncertainty_metrics.csv saved at:")
print(UNCERTAINTY_METRICS_CSV)
print("OK - uncertainty_policy.md saved at:")
print(UNCERTAINTY_POLICY_MD)
display(pd.DataFrame([policy_row]))
